# Dark Store Inventory Intelligence System
**Stockout Prediction + Dynamic Reorder Point**

This notebook covers the complete ML pipeline:
1. Data Cleaning
2. Feature Engineering
3. Train-Test Split
4. Encoding
5. Model Training & Evaluation (Model A + Model B)
6. Combined Pipeline
7. Saving Models

## 1. Imports

In [34]:
import pandas as pd
import numpy as np
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    confusion_matrix, classification_report,
    mean_squared_error, mean_absolute_error, r2_score
)
from xgboost import XGBClassifier, XGBRegressor

## 2. Load Data

In [35]:
# Update this path to where your CSV file is located
DATA_PATH = r'Data/dark_store_inventory_raw.csv'

df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
df.head()

Shape: (82400, 19)


,date,store_id,city,area,area_type,income_level,sku_id,category,price,discount_pct,weather,is_weekend,is_holiday,raw_demand,units_sold,inventory_level,lead_time_days,order_placed,order_quantity
0,2023-01-22,DS_PUN_02,Pune,Hadapsar,Residential,Medium,DETERGENT_1KG,Household,110,5.0,Cloudy,1,0,34,34,89,3.0,0,0
1,2022-11-19,DS_PUN_02,Pune,Hadapsar,Residential,Medium,ONION_1KG,Vegetables,40,5.0,Sunny,1,0,111,111,143,1.0,0,0
2,2023-11-01,DS_MUM_02,Mumbai,Dharavi,Mixed,Low,CHIPS_LAYS,Snacks,20,0.0,Cloudy,0,0,53,53,127,2.0,0,0
3,2023-01-08,DS_BLR_01,Bengaluru,Koramangala,Commercial,High,BUTTER_100G,Dairy,55,0.0,Sunny,1,0,18,18,79,1.0,0,0
4,2022-06-27,DS_MUM_02,Mumbai,Dharavi,Mixed,Low,DIAPERS_SM,BabyCare,350,0.0,Cloudy,0,0,2,2,36,3.0,0,0


## 3. Data Cleaning

### 3.1 Check Data Types

In [36]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 82400 entries, 0 to 82399
Data columns (total 19 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   date             82400 non-null  object 
 1   store_id         82400 non-null  object 
 2   city             82400 non-null  object 
 3   area             82400 non-null  object 
 4   area_type        82400 non-null  object 
 5   income_level     81576 non-null  object 
 6   sku_id           82400 non-null  object 
 7   category         82400 non-null  object 
 8   price            82400 non-null  int64  
 9   discount_pct     79928 non-null  float64
 10  weather          80750 non-null  object 
 11  is_weekend       82400 non-null  int64  
 12  is_holiday       82400 non-null  int64  
 13  raw_demand       82400 non-null  int64  
 14  units_sold       82400 non-null  int64  
 15  inventory_level  82400 non-null  int64  
 16  lead_time_days   80337 non-null  float64
 17  order_placed

### 3.2 Fix Date Column
The date column has mixed formats (yyyy-mm-dd and dd/mm/yyyy). We unify them.

In [37]:
df['date'] = pd.to_datetime(df['date'], errors='coerce', format='mixed')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 82400 entries, 0 to 82399
Data columns (total 19 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   date             82400 non-null  datetime64[ns]
 1   store_id         82400 non-null  object        
 2   city             82400 non-null  object        
 3   area             82400 non-null  object        
 4   area_type        82400 non-null  object        
 5   income_level     81576 non-null  object        
 6   sku_id           82400 non-null  object        
 7   category         82400 non-null  object        
 8   price            82400 non-null  int64         
 9   discount_pct     79928 non-null  float64       
 10  weather          80750 non-null  object        
 11  is_weekend       82400 non-null  int64         
 12  is_holiday       82400 non-null  int64         
 13  raw_demand       82400 non-null  int64         
 14  units_sold       82400 non-null  int64

### 3.3 Sort Data
Mandatory before any rolling window, lag, or forward-fill operation.
Sort order: store_id → sku_id → date ensures each group is processed correctly.

In [38]:
df.sort_values(['store_id', 'sku_id', 'date'], inplace=True)
df.reset_index(drop=True, inplace=True)
print("Data sorted. Shape:", df.shape)

Data sorted. Shape: (82400, 19)


### 3.4 Check Missing Values

In [39]:
df.isnull().sum()[df.isnull().sum() > 0]

income_level       824
discount_pct      2472
weather           1650
lead_time_days    2063
dtype: int64

### 3.5 Fill Missing Values

In [40]:
# income_level is a store property — same value for all rows of that store
# Fill nulls using the most common income_level for each store_id
store_income_map = (
    df[df['income_level'].notna()]
    .groupby('store_id')['income_level']
    .agg(lambda x: x.mode()[0])
)
df['income_level'] = df['income_level'].fillna(df['store_id'].map(store_income_map))

# discount_pct: NaN means no discount was applied
df['discount_pct'] = df['discount_pct'].fillna(0)

# weather depends on city + season
# derive season first, then fill missing weather using city + season mode
def get_season(month):
    if month in [12, 1, 2]:   return 'Winter'
    elif month in [3, 4, 5]:  return 'Summer'
    elif month in [6, 7, 8, 9]: return 'Monsoon'
    else:                      return 'Festive'

df['season'] = df['date'].dt.month.apply(get_season)

df['weather'] = (
    df.groupby(['city', 'season'])['weather']
    .transform(lambda x: x.fillna(x.mode()[0] if not x.mode().empty else 'Sunny'))
)

# lead_time_days: fill with median per SKU (supplier property)
df['lead_time_days'] = (
    df.groupby('sku_id')['lead_time_days']
    .transform(lambda x: x.fillna(x.median()))
)

# Verify all nulls are fixed
print("Remaining nulls:")
print(df.isnull().sum()[df.isnull().sum() > 0])
print("All nulls fixed!" if df.isnull().sum().sum() == 0 else "Some nulls remain.")

Remaining nulls:
Series([], dtype: int64)
All nulls fixed!


### 3.6 Check and Remove Duplicates

In [41]:
print("Duplicate rows:", df.duplicated().sum())
df.drop_duplicates(inplace=True)
df.reset_index(drop=True, inplace=True)
print("After removing duplicates. Shape:", df.shape)

Duplicate rows: 300
After removing duplicates. Shape: (82100, 20)


### 3.7 Fix Impossible Values and Outliers

In [42]:
# Check describe for outliers
df.describe()

,date,price,discount_pct,is_weekend,is_holiday,raw_demand,units_sold,inventory_level,lead_time_days,order_placed,order_quantity
count,82100,82100.000000,82100.000000,82100.000000,82100.000000,82100.000000,82100.000000,82100.000000,82100.000000,82100.000000,82100.000000
mean,2023-02-15 05:43:33.924482304,74.824129,1.472351,0.287454,0.028015,47.748636,47.433752,213.334823,1.746845,0.136760,47.435043
min,2022-01-01 00:00:00,0.000000,0.000000,0.000000,0.000000,0.000000,-187.000000,0.000000,1.000000,0.000000,0.000000
25%,2022-07-25 00:00:00,30.000000,0.000000,0.000000,0.000000,25.000000,25.000000,81.000000,1.000000,0.000000,0.000000
50%,2023-02-15 00:00:00,45.000000,0.000000,0.000000,0.000000,41.000000,40.000000,156.000000,2.000000,0.000000,0.000000
75%,2023-09-08 00:00:00,85.000000,0.000000,1.000000,0.000000,63.000000,62.000000,278.000000,2.000000,0.000000,0.000000
max,2024-12-03 00:00:00,350.000000,20.000000,1.000000,1.000000,329.000000,329.000000,9999.000000,4.000000,1.000000,1658.000000
std,NaN,77.808796,3.421708,0.452578,0.165016,31.110541,31.308447,350.575446,0.786666,0.343596,140.982339


In [43]:
# price = 0 is not possible — replace with SKU median price
print("Zero prices:", (df['price'] == 0).sum())
df['price'] = df['price'].replace(0, np.nan)
df['price'] = df.groupby('sku_id')['price'].transform(lambda x: x.fillna(x.median()))

# units_sold < 0 is a data entry error — fix with absolute value
print("Negative units_sold:", (df['units_sold'] < 0).sum())
df['units_sold'] = df['units_sold'].abs()

# inventory_level = 9999 is a system glitch
# Replace with NaN then forward-fill within each store+SKU group
print("Inventory outliers (9999):", (df['inventory_level'] == 9999).sum())
df['inventory_level'] = df['inventory_level'].replace(9999, np.nan)
df['inventory_level'] = (
    df.groupby(['store_id', 'sku_id'])['inventory_level']
    .transform(lambda x: x.ffill())
)

print("\nFinal check:")
print(df.describe())

Zero prices: 200
Negative units_sold: 150
Inventory outliers (9999): 80

Final check:
                                date         price  discount_pct  \
count                          82100  82100.000000  82100.000000   
mean   2023-02-15 05:43:33.924482304     75.000000      1.472351   
min              2022-01-01 00:00:00     14.000000      0.000000   
25%              2022-07-25 00:00:00     30.000000      0.000000   
50%              2023-02-15 00:00:00     47.500000      0.000000   
75%              2023-09-08 00:00:00     87.500000      0.000000   
max              2024-12-03 00:00:00    350.000000     20.000000   
std                              NaN     77.799574      3.421708   

         is_weekend    is_holiday    raw_demand    units_sold  \
count  82100.000000  82100.000000  82100.000000  82100.000000   
mean       0.287454      0.028015     47.748636     47.595895   
min        0.000000      0.000000      0.000000      0.000000   
25%        0.000000      0.000000     25.

### 3.8 Final Cleaning Check

In [44]:
print("Shape after cleaning:", df.shape)
print("Nulls remaining:", df.isnull().sum().sum())
df.head()

Shape after cleaning: (82100, 20)
Nulls remaining: 0


,date,store_id,city,area,area_type,income_level,sku_id,category,price,discount_pct,weather,is_weekend,is_holiday,raw_demand,units_sold,inventory_level,lead_time_days,order_placed,order_quantity,season
0,2022-01-01,DS_BLR_01,Bengaluru,Koramangala,Commercial,High,ATTA_5KG,Staples,220.0,10.0,Sunny,1,0,20,20,215.0,2.0,0,0,Winter
1,2022-01-02,DS_BLR_01,Bengaluru,Koramangala,Commercial,High,ATTA_5KG,Staples,220.0,10.0,Sunny,1,0,28,28,187.0,2.0,0,0,Winter
2,2022-01-03,DS_BLR_01,Bengaluru,Koramangala,Commercial,High,ATTA_5KG,Staples,220.0,0.0,Foggy,0,0,12,12,175.0,2.0,0,0,Winter
3,2022-01-04,DS_BLR_01,Bengaluru,Koramangala,Commercial,High,ATTA_5KG,Staples,220.0,0.0,Cloudy,0,0,23,23,152.0,2.0,0,0,Winter
4,2022-01-05,DS_BLR_01,Bengaluru,Koramangala,Commercial,High,ATTA_5KG,Staples,220.0,0.0,Sunny,0,0,23,23,129.0,3.0,0,0,Winter


## 4. Feature Engineering

> **Important:** All features are derived from historical data only. No future information is used.

### 4.1 Date-Based Features

In [45]:
df['day_of_week']    = df['date'].dt.dayofweek          # 0=Monday, 6=Sunday
df['month']          = df['date'].dt.month
df['is_month_start'] = df['date'].dt.is_month_start.astype(int)
df['is_month_end']   = df['date'].dt.is_month_end.astype(int)
df['is_ipl_season']  = df['month'].isin([4, 5]).astype(int)  # Apr-May: beverages/snacks spike

print("Date features added.")
df[['date', 'day_of_week', 'month', 'is_month_start', 'is_month_end', 'is_ipl_season']].head()

Date features added.


,date,day_of_week,month,is_month_start,is_month_end,is_ipl_season
0,2022-01-01,5,1,1,0,0
1,2022-01-02,6,1,0,0,0
2,2022-01-03,0,1,0,0,0
3,2022-01-04,1,1,0,0,0
4,2022-01-05,2,1,0,0,0


### 4.2 Lost Sales
Units of demand that could not be fulfilled due to stockout.

In [46]:
# lost_sales = how many units were demanded but not available
# clip at 0 to avoid negatives caused by data noise
df['lost_sales'] = (df['raw_demand'] - df['units_sold']).clip(lower=0)

print("Lost sales sample:")
df[['raw_demand', 'units_sold', 'lost_sales']].head(10)

Lost sales sample:


,raw_demand,units_sold,lost_sales
0,20,20,0
1,28,28,0
2,12,12,0
3,23,23,0
4,23,23,0
5,19,19,0
6,18,18,0
7,41,41,0
8,38,38,0
9,21,21,0


### 4.3 Rolling Demand Features
Calculated per store+SKU group to capture demand trends.

In [47]:
grp = df.groupby(['store_id', 'sku_id'])['units_sold']

df['rolling_avg_7d']  = grp.transform(lambda x: x.rolling(7,  min_periods=1).mean())
df['rolling_avg_30d'] = grp.transform(lambda x: x.rolling(30, min_periods=1).mean())
df['rolling_std_7d']  = grp.transform(lambda x: x.rolling(7,  min_periods=1).std().fillna(0))
df['rolling_std_30d'] = grp.transform(lambda x: x.rolling(30, min_periods=1).std().fillna(0))

print("Rolling features added.")
df[['rolling_avg_7d', 'rolling_avg_30d', 'rolling_std_7d', 'rolling_std_30d']].describe()

Rolling features added.


,rolling_avg_7d,rolling_avg_30d,rolling_std_7d,rolling_std_30d
count,82100.000000,82100.000000,82100.000000,82100.000000
mean,47.610754,47.635495,14.487996,15.055067
std,26.719881,26.141385,9.161165,8.204265
min,4.285714,4.500000,0.000000,0.000000
25%,27.142857,27.500000,7.934254,8.911829
50%,42.571429,43.200000,12.188988,13.264875
75%,61.714286,61.700000,18.547237,19.063385
max,169.714286,158.000000,98.999519,60.572897


### 4.4 Lag Features
Sales and inventory from past days — captures momentum.

In [48]:
grp_sales = df.groupby(['store_id', 'sku_id'])['units_sold']
grp_inv   = df.groupby(['store_id', 'sku_id'])['inventory_level']

df['lag_sales_1d']     = grp_sales.transform(lambda x: x.shift(1))
df['lag_sales_7d']     = grp_sales.transform(lambda x: x.shift(7))
df['lag_sales_14d']    = grp_sales.transform(lambda x: x.shift(14))
df['lag_inventory_1d'] = grp_inv.transform(lambda x: x.shift(1))

# Drop first 14 days of each store+SKU — incomplete lag windows
lag_cols = ['lag_sales_1d', 'lag_sales_7d', 'lag_sales_14d', 'lag_inventory_1d']
df.dropna(subset=lag_cols, inplace=True)
df.reset_index(drop=True, inplace=True)

print("Lag features added. Shape after dropping incomplete rows:", df.shape)

Lag features added. Shape after dropping incomplete rows: (80700, 34)


### 4.5 Demand Spike Flag

In [49]:
# 1 if today's sales are more than 1.5x the 7-day average
df['sales_spike_flag'] = (df['units_sold'] > 1.5 * df['rolling_avg_7d']).astype(int)

print("Spike events:", df['sales_spike_flag'].sum())

Spike events: 5180


### 4.6 Safety Stock and Reorder Point (Target B)

**Formula:**
```
safety_stock = 1.65 × √(lead_time × demand_std²)
reorder_point = (avg_demand × lead_time) + safety_stock
```

Z = 1.65 corresponds to 95% service level (standard inventory science).

In [50]:
Z = 1.65  # 95% service level

# NOTE: formula uses rolling_std_7d (demand variability), NOT rolling_avg_7d
# A common mistake is to use the average here — that would be wrong
df['safety_stock'] = (Z * np.sqrt(df['lead_time_days'] * df['rolling_std_7d'] ** 2)).round(0).astype(int)

# reorder_point: trigger a reorder when stock falls to this level
df['reorder_point'] = (df['rolling_avg_7d'] * df['lead_time_days'] + df['safety_stock']).round(0).astype(int)

print("Reorder point sample:")
df[['rolling_avg_7d', 'lead_time_days', 'safety_stock', 'reorder_point']].head(10)

Reorder point sample:


,rolling_avg_7d,lead_time_days,safety_stock,reorder_point
0,20.714286,2.0,21,62
1,18.285714,2.0,11,48
2,18.142857,2.0,11,47
3,20.714286,2.0,15,56
4,21.000000,2.0,15,57
5,20.285714,2.0,16,57
6,20.142857,3.0,20,80
7,21.000000,3.0,23,86
8,23.857143,2.0,26,74
9,23.571429,2.0,26,73


### 4.7 Stockout Flag
A row is a stockout if current inventory is critically low (≤ 20% of average daily demand).

In [51]:
df['stockout_flag'] = (df['inventory_level'] <= (0.2 * df['rolling_avg_7d'])).astype(int)

print("Stockout flag distribution:")
print(df['stockout_flag'].value_counts())

Stockout flag distribution:
stockout_flag
0    79550
1     1150
Name: count, dtype: int64


### 4.8 Target Variable A: stockout_in_48hrs
Looks 2 days forward — if stockout_flag = 1 in current row or next 2 rows, label = 1.

In [52]:
df['stockout_in_48hrs'] = df.groupby(['store_id', 'sku_id'])['stockout_flag'].transform(
    lambda x: (
        (x == 1) |
        (x.shift(-1).fillna(0).astype(int) == 1) |
        (x.shift(-2).fillna(0).astype(int) == 1)
    )
).astype(int)

print("stockout_in_48hrs distribution:")
print(df['stockout_in_48hrs'].value_counts())
print()
print("Positive rate:", df['stockout_in_48hrs'].mean().round(4) * 100, "%")

stockout_in_48hrs distribution:
stockout_in_48hrs
0    77344
1     3356
Name: count, dtype: int64

Positive rate: 4.16 %


## 5. Train-Test Split

> **Rule:** Never use random split on time-series data. We split by date.
> Train: Jan 2022 – Sep 2023 | Test: Oct 2023 – Mar 2024

In [53]:
CUTOFF = '2023-09-30'

df_train = df[df['date'] <= CUTOFF].copy()
df_test  = df[df['date'] >  CUTOFF].copy()

print(f"Train: {len(df_train):,} rows ({df_train['date'].min().date()} to {df_train['date'].max().date()})")
print(f"Test:  {len(df_test):,} rows ({df_test['date'].min().date()} to {df_test['date'].max().date()})")

# Keep raw test set for baseline comparison (needs days_of_stock_remaining)
x_test_raw = df_test.copy()

Train: 62,385 rows (2022-01-13 to 2023-09-30)
Test:  18,315 rows (2023-10-01 to 2024-12-03)


## 6. Encoding

Two types:
- **Ordinal:** income_level has a natural order (Low < Medium < High) → manual map
- **Nominal:** store_id, sku_id, city etc. have no order → LabelEncoder

> **Critical:** Fit encoders on train only. Transform test using the same fitted encoder.
> Fitting separately on test gives different integer codes — the model will misread them.

In [54]:
# Ordinal encoding — manual map because order matters
INCOME_MAP = {'Low': 0, 'Medium': 1, 'High': 2}
df_train['income_level'] = df_train['income_level'].map(INCOME_MAP)
df_test['income_level']  = df_test['income_level'].map(INCOME_MAP)

# Nominal encoding — fit on train, transform both
NOMINAL_COLS = ['store_id', 'city', 'area', 'area_type', 'sku_id', 'category', 'weather', 'season']

# Store a separate encoder per column so we can reuse them in the app
encoders = {}
for col in NOMINAL_COLS:
    le = LabelEncoder()
    df_train[col] = le.fit_transform(df_train[col].astype(str))
    # Use transform (not fit_transform) on test — same mapping as train
    df_test[col]  = le.transform(df_test[col].astype(str))
    encoders[col] = le   # save for use in app.py

print("Encoding done.")
print("Encoder keys:", list(encoders.keys()))

Encoding done.
Encoder keys: ['store_id', 'city', 'area', 'area_type', 'sku_id', 'category', 'weather', 'season']


## 7. Feature Selection

Drop columns that cannot be used as model inputs:

In [55]:
# Drop date — already extracted everything from it
df_train.drop(columns=['date'], errors='ignore', inplace=True)
df_test.drop(columns=['date'],  errors='ignore', inplace=True)

DROP_COLS = [
    'raw_demand',        # future information — model can't know demand before the day ends
    'order_quantity',    # consequence of reorder decision, not a predictor
    'stockout_flag',     # too directly related to target — leakage risk
    'lost_sales',        # derived from raw_demand which is dropped
    'safety_stock',      # intermediate calculation, reorder_point already encodes it
    'reorder_point',     # Target B — not an input feature
    'stockout_in_48hrs', # Target A — not an input feature
    'days_of_stock_remaining',  # removed: almost directly encodes the answer
]

FEATURE_COLS = [c for c in df_train.columns if c not in DROP_COLS]

x_train = df_train[FEATURE_COLS]
x_test  = df_test[FEATURE_COLS]

y_train_stockout = df_train['stockout_in_48hrs']
y_test_stockout  = df_test['stockout_in_48hrs']

y_train_reorder  = df_train['reorder_point']
y_test_reorder   = df_test['reorder_point']

print("Feature count:", len(FEATURE_COLS))
print("Features:", FEATURE_COLS)
print()
print("x_train:", x_train.shape, "| x_test:", x_test.shape)

Feature count: 31
Features: ['store_id', 'city', 'area', 'area_type', 'income_level', 'sku_id', 'category', 'price', 'discount_pct', 'weather', 'is_weekend', 'is_holiday', 'units_sold', 'inventory_level', 'lead_time_days', 'order_placed', 'season', 'day_of_week', 'month', 'is_month_start', 'is_month_end', 'is_ipl_season', 'rolling_avg_7d', 'rolling_avg_30d', 'rolling_std_7d', 'rolling_std_30d', 'lag_sales_1d', 'lag_sales_7d', 'lag_sales_14d', 'lag_inventory_1d', 'sales_spike_flag']

x_train: (62385, 31) | x_test: (18315, 31)


## 8. Baselines

Always build a rule-based baseline before ML. If your ML model can't beat a simple rule, something is wrong.

In [56]:
def baseline_stockout(X_raw, threshold_days=2.0):
    """
    Rule: flag as stockout risk if days_of_stock_remaining < threshold.
    This is what a store manager would do manually without any ML.
    """
    days_remaining = X_raw['inventory_level'] / X_raw['rolling_avg_7d'].replace(0, np.nan)
    return (days_remaining < threshold_days).fillna(0).astype(int).values


def baseline_reorder(X_raw):
    """
    Formula-only reorder point prediction (no ML).
    Difference from model: uses current-day rolling values directly,
    while the target was computed at time of dataset creation with slightly
    different window states — hence baseline shows high error.
    """
    Z = 1.65
    safety = Z * np.sqrt(X_raw['lead_time_days'] * X_raw['rolling_std_7d'] ** 2)
    return (X_raw['rolling_avg_7d'] * X_raw['lead_time_days'] + safety).round(0).values


# Run baselines on test set
baseline_pred_stockout = baseline_stockout(x_test_raw)
baseline_pred_reorder  = baseline_reorder(x_test_raw)

print("Baseline Stockout Prediction:")
print("  Recall:", round(recall_score(y_test_stockout, baseline_pred_stockout), 4))
print("  F1    :", round(f1_score(y_test_stockout, baseline_pred_stockout), 4))
print()
print("Baseline Reorder Point:")
print("  RMSE  :", round(np.sqrt(mean_squared_error(y_test_reorder, baseline_pred_reorder)), 2))
print("  MAE   :", round(mean_absolute_error(y_test_reorder, baseline_pred_reorder), 2))

Baseline Stockout Prediction:
  Recall: 0.7731
  F1    : 0.2593

Baseline Reorder Point:
  RMSE  : 0.5
  MAE   : 0.25


## 9. Model A — Stockout Prediction (XGBoost Classifier)

**Target:** `stockout_in_48hrs` (0 or 1)
**Primary metric:** Recall — missing a real stockout is worse than a false alarm

In [57]:
# Check class imbalance
print("Class distribution in training set:")
print(y_train_stockout.value_counts())
print()

neg = (y_train_stockout == 0).sum()
pos = (y_train_stockout == 1).sum()
scale_pos_weight = neg / pos   # gives more weight to minority class (stockouts)
print(f"scale_pos_weight: {scale_pos_weight:.2f} (neg={neg}, pos={pos})")

Class distribution in training set:
stockout_in_48hrs
0    59787
1     2598
Name: count, dtype: int64

scale_pos_weight: 23.01 (neg=59787, pos=2598)


In [58]:
model_stockout = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    scale_pos_weight=scale_pos_weight,  # handles class imbalance
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='logloss',
    verbosity=0,
    use_label_encoder=False,
)
model_stockout.fit(x_train, y_train_stockout)
print("Model A trained.")

Model A trained.


### 9.1 Evaluate Model A

In [59]:
y_pred_stockout = model_stockout.predict(x_test)
y_prob_stockout = model_stockout.predict_proba(x_test)[:, 1]

print("=" * 50)
print("MODEL A — STOCKOUT PREDICTION RESULTS")
print("=" * 50)
print(f"Precision : {precision_score(y_test_stockout, y_pred_stockout):.4f}")
print(f"Recall    : {recall_score(y_test_stockout, y_pred_stockout):.4f}  ← primary metric")
print(f"F1 Score  : {f1_score(y_test_stockout, y_pred_stockout):.4f}")
print()
print("Confusion Matrix:")
print(confusion_matrix(y_test_stockout, y_pred_stockout))
print()
print("Classification Report:")
print(classification_report(y_test_stockout, y_pred_stockout))
print()
print("Baseline comparison:")
print(f"  Baseline Recall : {recall_score(y_test_stockout, baseline_pred_stockout):.4f}")
print(f"  Baseline F1     : {f1_score(y_test_stockout, baseline_pred_stockout):.4f}")

MODEL A — STOCKOUT PREDICTION RESULTS
Precision : 0.2757
Recall    : 0.7573  ← primary metric
F1 Score  : 0.4042

Confusion Matrix:
[[16049  1508]
 [  184   574]]

Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.91      0.95     17557
           1       0.28      0.76      0.40       758

    accuracy                           0.91     18315
   macro avg       0.63      0.84      0.68     18315
weighted avg       0.96      0.91      0.93     18315


Baseline comparison:
  Baseline Recall : 0.7731
  Baseline F1     : 0.2593


### 9.2 Feature Importance — Model A

In [60]:
feature_imp_stockout = pd.Series(
    model_stockout.feature_importances_,
    index=x_test.columns
).sort_values(ascending=False)

print("Top 10 features driving stockout prediction:")
print(feature_imp_stockout.head(10))

Top 10 features driving stockout prediction:
inventory_level     0.180471
order_placed        0.086281
lead_time_days      0.079222
rolling_avg_30d     0.059299
day_of_week         0.043213
lag_inventory_1d    0.038064
rolling_avg_7d      0.037175
units_sold          0.034122
rolling_std_30d     0.029276
is_weekend          0.025865
dtype: float32


## 10. Model B — Reorder Point Prediction (XGBoost Regressor)

**Target:** `reorder_point` (numeric, in units)
**Metrics:** RMSE, MAE, R²

In [61]:
model_reorder = XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbosity=0,
)
model_reorder.fit(x_train, y_train_reorder)
print("Model B trained.")

Model B trained.


### 10.1 Evaluate Model B

In [62]:
y_pred_reorder = model_reorder.predict(x_test).clip(min=0)

print("=" * 50)
print("MODEL B — REORDER POINT RESULTS")
print("=" * 50)
print(f"RMSE : {np.sqrt(mean_squared_error(y_test_reorder, y_pred_reorder)):.4f}")
print(f"MAE  : {mean_absolute_error(y_test_reorder, y_pred_reorder):.4f}")
print(f"R²   : {r2_score(y_test_reorder, y_pred_reorder):.4f}")
print()
print("Baseline comparison:")
print(f"  Baseline RMSE : {np.sqrt(mean_squared_error(y_test_reorder, baseline_pred_reorder)):.4f}")
print(f"  Baseline MAE  : {mean_absolute_error(y_test_reorder, baseline_pred_reorder):.4f}")

MODEL B — REORDER POINT RESULTS
RMSE : 2.0837
MAE  : 1.1547
R²   : 0.9980

Baseline comparison:
  Baseline RMSE : 0.4977
  Baseline MAE  : 0.2477


### 10.2 Feature Importance — Model B

In [63]:
feature_imp_reorder = pd.Series(
    model_reorder.feature_importances_,
    index=x_test.columns
).sort_values(ascending=False)

print("Top 10 features driving reorder point prediction:")
print(feature_imp_reorder.head(10))

Top 10 features driving reorder point prediction:
price              0.487864
lead_time_days     0.175976
rolling_std_7d     0.139471
rolling_avg_7d     0.077934
rolling_avg_30d    0.038737
rolling_std_30d    0.035659
category           0.014188
sku_id             0.007229
season             0.006220
income_level       0.004856
dtype: float32


## 11. Combined Pipeline

The two models work sequentially:
1. **Model A** checks if a SKU is at stockout risk
2. **Only if at risk:** Model B calculates the reorder point

This mirrors how an inventory manager would actually use the system.

In [64]:
def combined_pipeline(X, stockout_model, reorder_model, threshold=0.4):
    """
    Sequential pipeline:
    Step 1 — Model A predicts stockout probability for every row
    Step 2 — Model B predicts reorder point ONLY for at-risk rows

    Parameters:
        X               : encoded feature DataFrame
        stockout_model  : trained XGBClassifier
        reorder_model   : trained XGBRegressor
        threshold       : probability cutoff for flagging as at-risk

    Returns:
        DataFrame with stockout_probability, at_risk flag,
        reorder_point (only for at-risk rows), and action
    """
    # Step 1 — stockout probability for all rows
    stockout_probs = stockout_model.predict_proba(X)[:, 1]
    at_risk        = (stockout_probs >= threshold).astype(int)

    # Step 2 — reorder point only for at-risk rows
    reorder_preds = np.zeros(len(X))
    if at_risk.sum() > 0:
        reorder_preds[at_risk == 1] = reorder_model.predict(X[at_risk == 1]).clip(min=0).round(0)

    result = pd.DataFrame({
        'stockout_probability':      (stockout_probs * 100).round(1),
        'at_risk':                   at_risk,
        'recommended_reorder_point': np.where(at_risk == 1, reorder_preds, np.nan),
        'action':                    np.where(at_risk == 1, 'Order Now', 'Monitor'),
    })
    return result


# Test on first 10 rows of test set
sample_result = combined_pipeline(x_test.head(10), model_stockout, model_reorder)
print("Combined pipeline output (sample):")
print(sample_result)

Combined pipeline output (sample):
   stockout_probability  at_risk  recommended_reorder_point   action
0              9.900000        0                        NaN  Monitor
1              0.200000        0                        NaN  Monitor
2              0.100000        0                        NaN  Monitor
3              0.400000        0                        NaN  Monitor
4              2.800000        0                        NaN  Monitor
5             23.799999        0                        NaN  Monitor
6             31.000000        0                        NaN  Monitor
7              0.300000        0                        NaN  Monitor
8              0.100000        0                        NaN  Monitor
9              0.200000        0                        NaN  Monitor


## 12. Save Models

Save trained models, encoders, and feature column list to the `models/` folder.
The Streamlit app will load these directly — no need to retrain.

In [65]:
MODELS_DIR = 'models'
os.makedirs(MODELS_DIR, exist_ok=True)

joblib.dump(model_stockout, os.path.join(MODELS_DIR, 'model_stockout.pkl'))
joblib.dump(model_reorder,  os.path.join(MODELS_DIR, 'model_reorder.pkl'))
joblib.dump(encoders,       os.path.join(MODELS_DIR, 'encoders.pkl'))
joblib.dump(FEATURE_COLS,   os.path.join(MODELS_DIR, 'feature_cols.pkl'))
joblib.dump(INCOME_MAP,     os.path.join(MODELS_DIR, 'income_map.pkl'))

print("Models saved to:", MODELS_DIR)
print("Files:")
for f in os.listdir(MODELS_DIR):
    print(" ", f)

Models saved to: models
Files:
  encoders.pkl
  feature_cols.pkl
  income_map.pkl
  model_reorder.pkl
  model_stockout.pkl


## 13. Save Evaluation Metrics

Save Model A and Model B performance metrics to the `metrics/` folder as JSON and CSV.
The Streamlit app's **Model Performance** page reads these files directly — no need to retrain or re-run this notebook to view them.

In [66]:
from sklearn.metrics import accuracy_score, r2_score
import json
from datetime import datetime

METRICS_DIR = 'metrics'
os.makedirs(METRICS_DIR, exist_ok=True)

# ── Model A metrics — Stockout Classifier ────────────────────────
model_a_metrics = {
    "algorithm":          "XGBoost Classifier",
    "target":             "stockout_in_48hrs",
    "precision":          round(float(precision_score(y_test_stockout, y_pred_stockout)), 4),
    "recall":             round(float(recall_score(y_test_stockout, y_pred_stockout)), 4),
    "f1_score":           round(float(f1_score(y_test_stockout, y_pred_stockout)), 4),
    "baseline_precision": round(float(precision_score(y_test_stockout, baseline_pred_stockout, zero_division=0)), 4),
    "baseline_recall":    round(float(recall_score(y_test_stockout, baseline_pred_stockout)), 4),
    "baseline_f1":        round(float(f1_score(y_test_stockout, baseline_pred_stockout)), 4),
}

# ── Model B metrics — Reorder Point Regressor ────────────────────
model_b_metrics = {
    "algorithm":    "XGBoost Regressor",
    "target":       "reorder_point",
    "rmse":         round(float(np.sqrt(mean_squared_error(y_test_reorder, y_pred_reorder))), 4),
    "mae":          round(float(mean_absolute_error(y_test_reorder, y_pred_reorder)), 4),
    "r2_score":     round(float(r2_score(y_test_reorder, y_pred_reorder)), 4),
    "baseline_rmse":round(float(np.sqrt(mean_squared_error(y_test_reorder, baseline_pred_reorder))), 4),
    "baseline_mae": round(float(mean_absolute_error(y_test_reorder, baseline_pred_reorder)), 4),
    "baseline_r2":  round(float(r2_score(y_test_reorder, baseline_pred_reorder)), 4),
}

# ── Combine and save as JSON ─────────────────────────────────────
all_metrics = {
    "generated_at":  datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    "train_period":  "Jan 2022 - Sep 2023",
    "test_period":   "Oct 2023 - Mar 2024",
    "n_train_rows":  int(len(df_train)),
    "n_test_rows":   int(len(df_test)),
    "model_a":       model_a_metrics,
    "model_b":       model_b_metrics,
}

with open(os.path.join(METRICS_DIR, 'model_metrics.json'), 'w') as f:
    json.dump(all_metrics, f, indent=4)

print("Metrics saved to metrics/model_metrics.json")
print()
print("Model A:")
for k, v in model_a_metrics.items():
    if k not in ("algorithm", "target"):
        print(f"  {k}: {v}")
print()
print("Model B:")
for k, v in model_b_metrics.items():
    if k not in ("algorithm", "target"):
        print(f"  {k}: {v}")


Metrics saved to metrics/model_metrics.json

Model A:
  precision: 0.2757
  recall: 0.7573
  f1_score: 0.4042
  baseline_precision: 0.1558
  baseline_recall: 0.7731
  baseline_f1: 0.2593

Model B:
  rmse: 2.0837
  mae: 1.1547
  r2_score: 0.998
  baseline_rmse: 0.4977
  baseline_mae: 0.2477
  baseline_r2: 0.9999


## Done

Run `streamlit run app.py` from the project root to launch the dashboard.